In [1]:
import pickle
import matplotlib.pyplot as plt


import rospy
import numpy as np
import open3d.visualization
import torch
import matplotlib.pyplot as plt
from PIL import Image
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from tools import realsense
import cv2
import open3d
from tools import data_logging
import os
from geometry_msgs.msg import Pose, PoseStamped
from std_msgs.msg import Header
from scipy.spatial.transform import Rotation as R
from sensor_msgs.msg import PointCloud2,PointField,CameraInfo
from sensor_msgs.msg import Image as ImageMsg
from geometry_msgs.msg import TransformStamped,Transform
import sensor_msgs.point_cloud2 as pc2
import tf2_msgs.msg
import tf2_ros
from scipy.spatial.transform import Rotation
from threading import Lock
from cv_bridge import CvBridge
import inspect
import copy

%matplotlib tk

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/home/medusar/custom-python/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/home/medusar/custom-python/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/medusar/burhan/samvenv/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/medusar/burhan/samvenv/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
   

AttributeError: _ARRAY_API not found

In [2]:
FIELDS_XYZ = [
    PointField(name='x', offset=0, datatype=PointField.FLOAT32, count=1),
    PointField(name='y', offset=4, datatype=PointField.FLOAT32, count=1),
    PointField(name='z', offset=8, datatype=PointField.FLOAT32, count=1),
]

In [3]:
def show_mask(mask, ax, random_color=False, borders = True):
    if random_color:
        color = np.concatenate([rng.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30/255, 144/255, 255/255, 0.6])
    h, w = mask.shape[-2:]
    mask = mask.astype(np.uint8)
    mask_image =  mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    if borders:
        contours, _ = cv2.findContours(mask,cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE) 
        # Try to smooth contours
        contours = [cv2.approxPolyDP(contour, epsilon=0.01, closed=True) for contour in contours]
        mask_image = cv2.drawContours(mask_image, contours, -1, (1, 1, 1, 0.5), thickness=2) 
    ax.imshow(mask_image)
    return mask_image

def show_points(coords, labels, ax, marker_size=375):
    # import pdb
    # pdb.set_trace()
    pos_points = coords[labels>=1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)   

def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor=(0, 0, 0, 0), lw=2))

def add_mask(ax,mask,point_coords,input_labels,borders=True):
    masked_image=show_mask(mask,ax,borders=borders)
    if point_coords is not None:
        assert input_labels is not None
        show_points(point_coords, input_labels, plt.gca())
    return masked_image




def represent_pointcloud_in_frame(points,g_current2desired):
    R=g_current2desired[:3,:3]
    t=g_current2desired[:3,3]
    return points@R.T+t

# Convert the datatype of point cloud from Open3D to ROS PointCloud2 (XYZRGB only)
def convertCloudFromOpen3dToRos(points_in_camera, should_shift=True,world_base_frame=False, frame_id="object"):
    # Set "header"
    header = Header()
    # header.stamp = rospy.Time.now()
    header.frame_id = frame_id

    # Set "fields" and "cloud_data"
    if should_shift:
        #create "object frame"
        t = TransformStamped()
        if world_base_frame:
            t.header.frame_id = fixed_frame
        else:
            t.header.frame_id = camera_frame
        # t.header.stamp = rospy.Time.now()
        t.child_frame_id = frame_id
        if world_base_frame:
            points_in_world=represent_pointcloud_in_frame(points_in_camera,camera_to_world_g)
            shift=np.min(points_in_world,axis=0)
            points=points_in_world-shift
            t.transform.translation.x = shift[0]
            t.transform.translation.y = shift[1]
            t.transform.translation.z = shift[2]
        else:
            shift=np.min(points_in_camera,axis=0)
            distances = np.linalg.norm(points_in_camera, axis=1)
            shift2 = points_in_camera[np.argmin(distances)]
            points=points_in_camera-shift

            t.transform.translation.x = shift[0]
            t.transform.translation.y = shift[1]
            t.transform.translation.z = shift[2]

        t.transform.rotation.x = 0
        t.transform.rotation.y = 0
        t.transform.rotation.z = 0
        t.transform.rotation.w = 1
        min_debre=PoseStamped()
        min_debre.header.frame_id=camera_frame
        # min_debre.header.stamp=rospy.Time.now()
        min_debre.pose.position.x=shift[0]*0.9
        min_debre.pose.position.y=shift[1]*0.9
        min_debre.pose.position.z=shift[2]*0.9
        min_debre.pose.orientation.x=0
        min_debre.pose.orientation.y=0
        min_debre.pose.orientation.z=0
        min_debre.pose.orientation.w=1
        tfm = tf2_msgs.msg.TFMessage([t])
    else:
        shift=0
        points=points_in_camera

    fields=FIELDS_XYZ
    cloud_data=points

    # create ros_cloud
    return pc2.create_cloud(header, fields, cloud_data),shift

def convertCloudFromOpen3dToRosPubtf(points_in_camera, should_shift=True,world_base_frame=False, frame_id="object",camera_to_world_g=None):
    # Set "header"
    header = Header()
    # header.stamp = rospy.Time.now()
    header.frame_id = frame_id

    # Set "fields" and "cloud_data"
    if should_shift:
        #create "object frame"
        t = TransformStamped()
        if world_base_frame:
            t.header.frame_id = fixed_frame
        else:
            t.header.frame_id = camera_frame
        t.header.stamp = rospy.Time.now()
        t.child_frame_id = frame_id
        if world_base_frame:
            points_in_world=represent_pointcloud_in_frame(points_in_camera,camera_to_world_g)
            shift=np.min(points_in_world,axis=0)
            points=points_in_world

            shift=np.min(points_in_camera,axis=0)
            inverse_tf=np.linalg.inv(camera_to_world_g)
            t.transform.translation.x = inverse_tf[0,3]
            t.transform.translation.y = inverse_tf[1,3]
            t.transform.translation.z = inverse_tf[2,3]
            #convert to quaternion
            r = Rotation.from_matrix(inverse_tf[0:3,0:3])
            # print("inverting")
            q = r.as_quat()
            t.transform.rotation.x = q[0]
            t.transform.rotation.y = q[1]
            t.transform.rotation.z = q[2]
            t.transform.rotation.w = q[3]
        else:
            shift=np.min(points_in_camera,axis=0)
            # distances = np.linalg.norm(points_in_camera, axis=1)
            # shift2 = points_in_camera[np.argmin(distances)]
            points=points_in_camera-shift

            t.transform.translation.x = shift[0]
            t.transform.translation.y = shift[1]
            t.transform.translation.z = shift[2]

            t.transform.rotation.x = 0
            t.transform.rotation.y = 0
            t.transform.rotation.z = 0
            t.transform.rotation.w = 1


        # pub_debre_grab.publish(min_debre)
        tfm = tf2_msgs.msg.TFMessage([t])
        # print(t)
        # t2m = tf2_msgs.msg.TFMessage([t2])
        pub_tf.publish(tfm)
        # pub_tf.publish(t2m)
    else:
        shift=0
        points=points_in_camera

    fields=FIELDS_XYZ
    cloud_data=points

    # create ros_cloud
    return pc2.create_cloud(header, fields, cloud_data),shift , t

In [5]:
rospy.init_node("sam2_trest", anonymous=True)
cloud_pub = rospy.Publisher("/cloud_stitched", PointCloud2, queue_size=1, latch=True)
cloud_temp_pub = rospy.Publisher("/cloud_stitched_temp", PointCloud2, queue_size=1, latch=True)
pub_tf = rospy.Publisher("/tf", tf2_msgs.msg.TFMessage, queue_size=1)

In [6]:
# load pickle file
with open('captured_data_fullv2.pkl', 'rb') as f:
    data = pickle.load(f)
print(data.keys())
print(data[0].keys())
plt.imshow(data[0]['rgb'])

dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21])
dict_keys(['rgb', 'depth', 'rgb_stamp', 'depth_stamp', 'camera_info', 'camera_to_world', 'camera_to_world_R'])


In [7]:
camera_frame="camera_color_optical_frame"
fixed_frame="world"
depth_scale=1e-3#hardcoded value of mm per https://github.com/IntelRealSense/realsense-ros/issues/277#issuecomment-525676873
bridge=CvBridge()


device = torch.device("cuda")
torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
# turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
if torch.cuda.get_device_properties(0).major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

seed=3
background_distance=3

sam2_checkpoint = "checkpoints/sam2.1_hiera_tiny.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_t.yaml"

sam2 = build_sam2(model_cfg, sam2_checkpoint, device=device, apply_postprocessing=False)
predictor = SAM2ImagePredictor(sam2)


In [7]:
result_dict = {}
for i in data.keys():
# for i in range(0,15):    
    object=data[i]
    color_mat=object['rgb']
    depth_mat=object['depth']
    depth_intrinsic_mat=object['camera_info']
    depth_intrinsic_mat=np.array(depth_intrinsic_mat.K).reshape((3,3))


    color_rgb=cv2.cvtColor(color_mat,cv2.COLOR_BGR2RGB)
    image=Image.fromarray(color_rgb)

    predictor.set_image(image)

    rng=np.random.default_rng(seed)

    fig=plt.figure(figsize=(20, 20))
    ax=fig.gca()
    ax.imshow(image)
    ax.axis('off')

    item_idx=1
    print("Click on an object to prompt SAM2")
    pt=plt.ginput(1)
    # print(f"Current line number: {inspect.currentframe().f_lineno}")
    print(f"Clicked point: {pt}")
    # user=input("Enter 'e' to execute or anything else to quit: ")
    masks, scores, logits = predictor.predict(
        point_coords=pt,
        point_labels=[item_idx],
        multimask_output=True,
    )

    sorted_ind = np.argsort(scores)[::-1]
    masks = masks[sorted_ind]
    scores = scores[sorted_ind]
    logits = logits[sorted_ind]



    masked_image=add_mask(ax, masks[0], point_coords=np.array(pt), input_labels=np.array([item_idx],dtype=np.int64), borders=True)
    depth_scaled=(depth_mat*depth_scale).astype(np.float32)

    rgbd=realsense.o3d_rgbd_from_color_and_depth_arrays(masked_image,depth_scaled,False)
    o3dpcd=open3d.t.geometry.PointCloud.create_from_rgbd_image(rgbd, open3d.core.Tensor(depth_intrinsic_mat),depth_scale=1,depth_max=background_distance)

    masked_depth=depth_scaled.copy()
    bool_mask=masks[0].astype(np.bool)
    masked_depth[np.logical_not(bool_mask)]=2*background_distance
    masked_rgbd=realsense.o3d_rgbd_from_color_and_depth_arrays(masked_image,masked_depth,False)
    segmented_pcd=open3d.t.geometry.PointCloud.create_from_rgbd_image(masked_rgbd, open3d.core.Tensor(depth_intrinsic_mat),depth_scale=1,depth_max=background_distance)
    all_points_in_camera=o3dpcd.point.positions.numpy()
    segmented_points_in_camera=segmented_pcd.point.positions.numpy()

    cloud_source_msg,shift=convertCloudFromOpen3dToRos(segmented_points_in_camera,should_shift=True, frame_id="debris")
    full_cloud,_=convertCloudFromOpen3dToRos(all_points_in_camera,should_shift=False, frame_id=camera_frame)

    cloud_source_msg,shift,t=convertCloudFromOpen3dToRosPubtf(segmented_points_in_camera,should_shift=True, frame_id="debris")
    translation = np.array([t.transform.translation.x, t.transform.translation.y, t.transform.translation.z])
    rotation = np.array([t.transform.rotation.x, t.transform.rotation.y, t.transform.rotation.z, t.transform.rotation.w])

    # Convert quaternion to a rotation matrix
    rotation_matrix = R.from_quat(rotation).as_matrix()

    # Create the 4x4 transformation matrix
    transform_matrix = np.eye(4)
    transform_matrix[:3, :3] = rotation_matrix
    transform_matrix[:3, 3] = translation
    
    debris_to_world_g = np.linalg.inv(object['camera_to_world']) 


    t = TransformStamped()
    t.header.frame_id = fixed_frame
    t.header.stamp = rospy.Time.now()
    t.child_frame_id = "debris_inworld"
    t.transform.translation.x = debris_to_world_g[0, 3]
    t.transform.translation.y = debris_to_world_g[1, 3]
    t.transform.translation.z = debris_to_world_g[2, 3]
    rotation = R.from_matrix(debris_to_world_g[:3, :3])
    q = rotation.as_quat()
    t.transform.rotation.x = q[0]
    t.transform.rotation.y = q[1]
    t.transform.rotation.z = q[2]
    t.transform.rotation.w = q[3]
    tfm = tf2_msgs.msg.TFMessage([t])
    pub_tf.publish(tfm)

    # # Add a column of ones to the segmented points to make them homogeneous
    homogeneous_points = np.hstack((segmented_points_in_camera, np.ones((segmented_points_in_camera.shape[0], 1))))

    # Apply the transformation
    transformed_points = (homogeneous_points @ debris_to_world_g.T)[:, :3]

    result_dict[i] = {
        "segmented_points_in_camera": segmented_points_in_camera,
        "segmented_points_in_world": transformed_points,
        "debris_to_world_g": debris_to_world_g,
        "cloud_source_msg": cloud_source_msg,
        "full_cloud": full_cloud
    }

    # Create a PointCloud2 message for the transformed points
    cloud_msg = pc2.create_cloud(Header(stamp=rospy.Time.now(), frame_id=fixed_frame), FIELDS_XYZ, transformed_points)

    # rospy.sleep(10)

    allpts=result_dict[0]['segmented_points_in_world']
    for i in range(1,i):
        allpts=np.vstack((allpts,result_dict[i]['segmented_points_in_world']))
    print(allpts.shape)
    cloud_msg = pc2.create_cloud(Header(stamp=rospy.Time.now(), frame_id=fixed_frame), FIELDS_XYZ, allpts)
    cloud_temp_pub.publish(cloud_msg)
    # rospy.sleep(2)

    # close the figure
    plt.close(fig)

Click on an object to prompt SAM2


invalid command name "139969563011456process_stream_events"
    while executing
"139969563011456process_stream_events"
    ("after" script)
can't invoke "event" command: application has been destroyed
    while executing
"event generate $w <<ThemeChanged>>"
    (procedure "ttk::ThemeChanged" line 6)
    invoked from within
"ttk::ThemeChanged"


Clicked point: [(np.float64(345.8757920683661), np.float64(511.28995009420737))]
(25121, 3)
Click on an object to prompt SAM2
Clicked point: [(np.float64(332.85497240556344), np.float64(367.05933229085593))]
(25121, 3)
Click on an object to prompt SAM2
Clicked point: [(np.float64(282.7748967793998), np.float64(256.8831659132959))]
(50029, 3)
Click on an object to prompt SAM2
Clicked point: [(np.float64(216.66919695286379), np.float64(119.66375869760736))]
(76038, 3)
Click on an object to prompt SAM2
Clicked point: [(np.float64(295.7957164422023), np.float64(100.63332995966516))]
(98734, 3)
Click on an object to prompt SAM2
Clicked point: [(np.float64(355.89180719359877), np.float64(80.60129970919968))]
(121326, 3)
Click on an object to prompt SAM2
Clicked point: [(np.float64(486.10000382162434), np.float64(191.77906759928305))]
(135786, 3)
Click on an object to prompt SAM2
Clicked point: [(np.float64(571.2361323861027), np.float64(299.9520309517966))]
(161122, 3)
Click on an object to 

In [8]:
result_dict.keys()

dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21])

In [9]:
allpts=result_dict[0]['segmented_points_in_world']
for i in range(1,len(result_dict.keys())):
    allpts=np.vstack((allpts,result_dict[i]['segmented_points_in_world']))
cloud_msg = pc2.create_cloud(Header(stamp=rospy.Time.now(), frame_id=fixed_frame), FIELDS_XYZ, allpts)
cloud_temp_pub.publish(cloud_msg)

allpts1=result_dict[0]['segmented_points_in_world']
cloud_msg = pc2.create_cloud(Header(stamp=rospy.Time.now(), frame_id=fixed_frame), FIELDS_XYZ, allpts1)
cloud_pub.publish(cloud_msg)


print(allpts.shape)
print(allpts1.shape)

(491660, 3)
(25121, 3)
